In [2]:
from pathlib import Path
import sys

# 현재 notebook 위치에서 StockLens 프로젝트 루트 찾기
current = Path.cwd()

while current != current.parent:
    if (current / "src").is_dir():
        PROJECT_ROOT = current
        break
    current = current.parent
else:
    raise FileNotFoundError("StockLens 프로젝트 루트를 찾지 못했습니다.")

# 프로젝트 루트를 Python path에 추가
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("PROJECT_ROOT:", PROJECT_ROOT)
print("src exists:", (PROJECT_ROOT / "src").is_dir())

PROJECT_ROOT: /Users/yangjaehoon/Desktop/StockLens
src exists: True


In [3]:
import numpy as np
import pandas as pd

from src.data.dataset import build_combined_dataset, split_by_time
from src.ml.preprocessing import create_preprocessor
from src.feature_selection.elastic_net import run_elastic_net
from src.features.engineering import FEATURE_COLUMNS

In [4]:
from src.data.storage import HistoricalStorage

STOCK_CODES = (
    "000660",
    "005380",
    "005930",
    "035420",
    "035720",
)

storage = HistoricalStorage(PROJECT_ROOT / "data")

stock_bars = {}

for stock_code in STOCK_CODES:
    bars = storage.load_daily_bars(stock_code)
    stock_bars[stock_code] = bars

    print(
        f"{stock_code}: "
        f"{len(bars)} bars, "
        f"{bars[0].trade_date} ~ {bars[-1].trade_date}"
    )

000660: 601 bars, 2024-03-13 ~ 2026-09-01
005380: 601 bars, 2024-03-13 ~ 2026-09-01
005930: 601 bars, 2024-03-13 ~ 2026-09-01
035420: 601 bars, 2024-03-13 ~ 2026-09-01
035720: 601 bars, 2024-03-13 ~ 2026-09-01


In [5]:
dataset = build_combined_dataset(stock_bars)

print("Dataset shape:", dataset.shape)
print("Columns:", dataset.columns.tolist())

Dataset shape: (2685, 28)
Columns: ['trade_date', 'stock_code', 'return_1d', 'return_5d', 'return_10d', 'return_20d', 'intraday_return', 'high_low_range', 'gap', 'sma_5', 'sma_20', 'sma_60', 'price_to_sma_5', 'price_to_sma_20', 'price_to_sma_60', 'rsi_14', 'roc_10', 'roc_20', 'macd', 'macd_signal', 'macd_hist', 'volatility_5', 'volatility_20', 'atr_14', 'volume_change_1d', 'volume_sma_20', 'volume_ratio_20', 'target_return_5d']


In [6]:
splits = split_by_time(dataset)

train = splits.train
validation = splits.validation
test = splits.test

print("Train:", train.shape)
print("Validation:", validation.shape)
print("Test:", test.shape)

Train: (1895, 28)
Validation: (600, 28)
Test: (190, 28)


In [7]:
print("Train:")
print(train["trade_date"].min(), "~", train["trade_date"].max())

print("\nValidation:")
print(validation["trade_date"].min(), "~", validation["trade_date"].max())

print("\nTest:")
print(test["trade_date"].min(), "~", test["trade_date"].max())

Train:
2024-06-11 ~ 2025-12-30

Validation:
2026-01-02 ~ 2026-06-30

Test:
2026-07-01 ~ 2026-08-25


In [8]:
from src.data.dataset import TARGET_COLUMN

X_train = train[list(FEATURE_COLUMNS)]
y_train = train[TARGET_COLUMN]

X_val = validation[list(FEATURE_COLUMNS)]
y_val = validation[TARGET_COLUMN]

X_test = test[list(FEATURE_COLUMNS)]
y_test = test[TARGET_COLUMN]

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

print("X_val:", X_val.shape)
print("y_val:", y_val.shape)

print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

X_train: (1895, 25)
y_train: (1895,)
X_val: (600, 25)
y_val: (600,)
X_test: (190, 25)
y_test: (190,)


In [9]:
preprocessor = create_preprocessor()

train_processed = preprocessor.fit_transform(train)
validation_processed = preprocessor.transform(validation)
test_processed = preprocessor.transform(test)

In [10]:
X_train = train_processed[list(FEATURE_COLUMNS)]
X_val = validation_processed[list(FEATURE_COLUMNS)]
X_test = test_processed[list(FEATURE_COLUMNS)]

y_train = train_processed[TARGET_COLUMN]
y_val = validation_processed[TARGET_COLUMN]
y_test = test_processed[TARGET_COLUMN]

In [11]:
print("X_train:", X_train.shape)
print("X_val:", X_val.shape)
print("X_test:", X_test.shape)

print("\nMissing values:")
print("Train:", X_train.isna().sum().sum())
print("Validation:", X_val.isna().sum().sum())
print("Test:", X_test.isna().sum().sum())

X_train: (1895, 25)
X_val: (600, 25)
X_test: (190, 25)

Missing values:
Train: 0
Validation: 0
Test: 0
